In [1]:
import os
%pwd

'd:\\omar\\MLOps\\Chest-Cancer-Classification\\research'

In [2]:
os.chdir("d:/omar/MLOps/Chest-Cancer-Classification")
%pwd

'd:\\omar\\MLOps\\Chest-Cancer-Classification'

In [3]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [4]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories


In [5]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [6]:
import os
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
     
    def download_file(self)-> str:
        '''
        Fetch data from the url
        '''

        try: 
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")

            file_id = dataset_url.split("/")[-2]
            prefix = 'https://drive.google.com/uc?/export=download&id='
            gdown.download(prefix+file_id,zip_download_dir)

            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")

        except Exception as e:
            raise e
        
    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [7]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-24 02:23:59,691: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-24 02:23:59,705: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-24 02:23:59,707: INFO: common: created directory at: artifacts]
[2026-03-24 02:23:59,711: INFO: common: created directory at: artifacts/data_ingestion]
[2026-03-24 02:23:59,713: INFO: 36341839: Downloading data from https://drive.google.com/file/d/1OuHWGqwU5za4Fr5bpWpnniUE8XtFAF2C/view?usp=drive_link into file artifacts/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?/export=download&id=1OuHWGqwU5za4Fr5bpWpnniUE8XtFAF2C
From (redirected): https://drive.google.com/uc?%2Fexport=download&id=1OuHWGqwU5za4Fr5bpWpnniUE8XtFAF2C&confirm=t&uuid=cb658c0a-2e1c-45e3-9946-29b77cae3faa
To: d:\omar\MLOps\Chest-Cancer-Classification\artifacts\data_ingestion\data.zip
100%|██████████| 49.0M/49.0M [00:33<00:00, 1.47MB/s]

[2026-03-24 02:24:37,121: INFO: 36341839: Downloaded data from https://drive.google.com/file/d/1OuHWGqwU5za4Fr5bpWpnniUE8XtFAF2C/view?usp=drive_link into file artifacts/data_ingestion/data.zip]
